In [171]:
import pandas as pd 
import json
import os 
import sys 
import numpy as np 
import ast
import re

### From Scratch (DO NOT RUN ALWAYS)

In [174]:
path_to_survey = "../../../data/conv-trs/ecir-2026/human-eval"

In [175]:
with open(f"{path_to_survey}/surveys_collection_raw.json", "r") as fp: 
    data = json.load(fp)

In [176]:
metrics = ['relevance_match', 'popularity_mix_qid_0', 'sustainability_qid_0', 'diversity_qid_0']

In [177]:
comet = data[3]

In [178]:
count_unsures = 0 

for key, value in data[0]['flatAnswers'].items():
    if 'not_sure' in key: 
        count_unsures+=1

print(count_unsures)

0


In [179]:
MAPPING = {
    1.0: 2, 
    2.0: 1, 
    3.0: 0, 
    4.0: -1, 
    5.0: -2,
    'not_sure': -3, 
    0.0: -3, 
    -3.0: -3
}

In [180]:
humans = pd.read_csv(f"{path_to_survey}/survey_data_cleaned.csv")

In [181]:
parsed = []
persons = ['Wolfgang', 'Ewald']
cleaned_metrics = ['relevance', 'popularity_mix', 'diversity', 'sustainability']

for i, row in humans.iterrows(): 
    qid = row['query_id']
    query = row['query']
    
    for person in persons: 

        result = result = {
            "query_id": qid, 
            'query': query, 
            'annotator': person
        }
        for metric in cleaned_metrics: 
            col = f'{person}_{metric}_score'
            unsure = f'{person}_{metric}_not_sure'

            notsure = False
            parse_key = metric.split('_')[0]      


            try:
                if ast.literal_eval(row[unsure]):
                    notsure = True
                    result[parse_key] = MAPPING['not_sure']
            except ValueError as e: 
                if row[unsure]:
                    result[parse_key] = MAPPING['not_sure']
                    notsure = True
            
            if not notsure:
                try: 
                    if type(row[col]) == type('a'):
                        print(person)
                        map_key = float(ast.literal_eval(row[col]))
                    else: 
                        map_key = float(row[col])
                    
                    result[parse_key] = MAPPING[map_key]
                except Exception as e: 
                    print(f"Error: {e}; Row Index: {i};\nRow: {row}\n")
                    pass
            
            else: 
                pass
        
        result['feedback'] = None
        parsed.append(result)



In [182]:
df1 = pd.DataFrame(parsed)
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202 entries, 0 to 201
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   query_id        202 non-null    object
 1   query           202 non-null    object
 2   annotator       202 non-null    object
 3   relevance       202 non-null    int64 
 4   popularity      202 non-null    int64 
 5   diversity       202 non-null    int64 
 6   sustainability  202 non-null    int64 
 7   feedback        0 non-null      object
dtypes: int64(4), object(4)
memory usage: 12.8+ KB


### Parse JSON

In [172]:
def get_config_from_id(id):
    try:
        # pattern = r'c_[^_]+(?:_[^_]+)*?(?=_dim)'
        pattern = r'c_[^_]+(?:_[^_]+)*?(?=(?:_dim|_feedback))'
        return re.search(pattern, id).group(0)
    except Exception as e: 
        print(e, id)
        print(re.search(pattern, id))

def get_metric_from_id(id):
    try:
        return id.split("_")[8]
    except Exception as e: 
        # feedback
        return None

def parse_json(flatList, annotator='Comet [AI]'):
    results = []
    parsed_queries = []

    METRICS = ['relevance', 'diversity', 'popularity', 'sustainability']

    path_to_qs = f"../../../data/conv-trs/ecir-2026/selected_queries/filtered_queries.json"
    configs = pd.read_json(path_to_qs)

    temp_result = {}
    prev = None

    # sort dictionary 
    data_sorted = dict(sorted(flatList.items()))
    
    for key, value in data_sorted.items(): 
        if 'ranking' not in key: 
            continue 

        qid = get_config_from_id(key)
        if len(parsed_queries) == 0: 
            # start with the processing
            print(f"Starting the parser for qid: {qid}")
            prev = qid 
            parsed_queries.append(qid)
            temp_result['query_id'] = qid 
            temp_result['query'] = configs[configs['query_id'] == qid]['query_text'].values[0]
            temp_result['annotator'] = annotator
        
        if qid != prev: 
            # query has changed, update results 
            results.append(temp_result)
            print(f"Parsed. Appending dict for qid {prev}")

            prev = qid 
            temp_result = {}
            temp_result['query_id'] = qid 
            temp_result['query'] = configs[configs['query_id'] == qid]['query_text'].values[0]
            temp_result['annotator'] = annotator

        # still the same query
        if 'feedback' in key: 
            temp_result['feedback'] = value 
        else:  
            metric = get_metric_from_id(key)
            if 'not_sure' in key: 
                temp_result[metric] = MAPPING['not_sure']
            else: 
                temp_result[metric] = MAPPING[float(value)]

            print(f"processed metric: {metric} for qid: {qid}")

    print(f"Parsed. Appending dict for {qid}")
    results.append(temp_result)

    return results 


### For Other Annotators

In [189]:
# For other annotators 

def run(annotator_name):
    # access current human_eval_df 
    path_to_survey = "../../../data/conv-trs/ecir-2026/human-eval"

    try: 
        df = pd.read_csv(f"{path_to_survey}/survey_parsed.csv")
    except Exception as e: 
        # CSV doesn't exist - need to run the entire notebook
        print("No CSV found. Please parse the files from scratch!")
        return None
    
    with open(f"{path_to_survey}/surveys_collection_raw.json", "r") as fp: 
        responses = json.load(fp)
    
    data = None
    for item in responses: 
        if item['prolificId'] == annotator_name: 
            data = item['flatAnswers']
            break 
    
    if not data: 
        print("Annotator not found!")
        return None
    
    res = parse_json(data, annotator_name)
    res_df = pd.DataFrame(res)
    try:
        new_df = pd.concat([df, res_df])
    except Exception as e: 
        print(res)

    sorted_new = new_df.sort_values(by=['query_id']).reset_index(drop=True)
    print(f"Parsed data for {annotator_name}. Storing CSV")
    sorted_new.to_csv(f"{path_to_survey}/survey_parsed.csv", index=False)

In [190]:
run(annotator_name="Yas")

Starting the parser for qid: c_p_0_pop_high_hard
processed metric: diversity for qid: c_p_0_pop_high_hard
processed metric: popularity for qid: c_p_0_pop_high_hard
processed metric: relevance for qid: c_p_0_pop_high_hard
processed metric: sustainability for qid: c_p_0_pop_high_hard
processed metric: sustainability for qid: c_p_0_pop_high_hard
Parsed. Appending dict for qid c_p_0_pop_high_hard
processed metric: diversity for qid: c_p_0_pop_medium_easy
processed metric: popularity for qid: c_p_0_pop_medium_easy
processed metric: relevance for qid: c_p_0_pop_medium_easy
processed metric: sustainability for qid: c_p_0_pop_medium_easy
Parsed. Appending dict for qid c_p_0_pop_medium_easy
processed metric: diversity for qid: c_p_12_pop_low_sustainable
processed metric: popularity for qid: c_p_12_pop_low_sustainable
processed metric: relevance for qid: c_p_12_pop_low_sustainable
processed metric: sustainability for qid: c_p_12_pop_low_sustainable
Parsed. Appending dict for qid c_p_12_pop_low_s

UnboundLocalError: cannot access local variable 'new_df' where it is not associated with a value

In [184]:
print(path_to_survey)

../../../data/conv-trs/ecir-2026/human-eval


In [185]:
df = pd.read_csv(f"{path_to_survey}/survey_parsed.csv")

### Ignore

In [158]:
df.head()

,query_id,query,annotator,relevance,popularity,diversity,sustainability,feedback
0,c_p_68_pop_medium_sustainable,Budget-friendly European city break in April.,Wolfgang,-1,0,0,-1,None
1,c_p_68_pop_medium_sustainable,Budget-friendly European city break in April.,Ewald,-1,-1,0,-1,None
2,c_p_196_pop_high_sustainable,Cheap European city break in October with muse...,Wolfgang,2,1,0,1,None
3,c_p_196_pop_high_sustainable,Cheap European city break in October with muse...,Ewald,1,-1,1,1,None
4,c_p_9_pop_medium_sustainable,Walkable European city break in June with inte...,Wolfgang,2,0,0,0,None


In [161]:
sorted_df = df.sort_values(by=['query_id']).reset_index(drop=True)
sorted_df.head()

,query_id,query,annotator,relevance,popularity,diversity,sustainability,feedback
0,c_p_0_pop_high_easy,Suggest popular European cities with outdoor a...,Wolfgang,1,-1,-1,0,None
1,c_p_0_pop_high_easy,Suggest popular European cities with outdoor a...,Ewald,-1,-1,-1,-1,None
2,c_p_0_pop_high_easy,Suggest popular European cities with outdoor a...,Comet [AI],-1,1,-2,-1,QUERY 88 EVALUATION:\n\nQ1 - Better Match (Sco...
3,c_p_0_pop_high_hard,Best European cities for a luxurious culinary ...,Wolfgang,0,0,0,0,None
4,c_p_0_pop_high_hard,Best European cities for a luxurious culinary ...,Ewald,-2,-1,1,-1,None


In [166]:
sorted_df.head()

,query_id,query,annotator,relevance,popularity,diversity,sustainability,feedback
0,c_p_0_pop_high_easy,Suggest popular European cities with outdoor a...,Wolfgang,1,-1,-1,0,None
1,c_p_0_pop_high_easy,Suggest popular European cities with outdoor a...,Ewald,-1,-1,-1,-1,None
2,c_p_0_pop_high_easy,Suggest popular European cities with outdoor a...,Comet [AI],-1,1,-2,-1,QUERY 88 EVALUATION:\n\nQ1 - Better Match (Sco...
3,c_p_0_pop_high_hard,Best European cities for a luxurious culinary ...,Wolfgang,0,0,0,0,None
4,c_p_0_pop_high_hard,Best European cities for a luxurious culinary ...,Ewald,-2,-1,1,-1,None


In [168]:
sorted_df.to_csv(f"{path_to_survey}/survey_parsed.csv", index=False)